# Excercises 
# 1. Tune the network
Run the experiment below, explore the different parameters (see suggestions below) and study the result with tensorboard. 
Make a single page (1 a4) report of your findings. Use your visualisation skills to communicate your most important findings.

In [30]:
from mads_datasets import DatasetFactoryProvider, DatasetType

from mltrainer.preprocessors import BasePreprocessor
from mltrainer import imagemodels, Trainer, TrainerSettings, ReportTypes, metrics

import torch.optim as optim
from torch import nn
from tomlserializer import TOMLSerializer

We will be using `tomlserializer` to easily keep track of our experiments, and to easily save the different things we did during our experiments.
It can export things like settings and models to a simple `toml` file, which can be easily shared, checked and modified.

First, we need the data. 

In [ ]:
fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
preprocessor = BasePreprocessor()
streamers = fashionfactory.create_datastreamer(batchsize=64, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()

2025-09-17 20:46:20.988 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\j.nagelhout\.cache\mads_datasets\fashionmnist
2025-09-17 20:46:20.990 | INFO     | mads_datasets.base:download_data:124 - File already exists at C:\Users\j.nagelhout\.cache\mads_datasets\fashionmnist\fashionmnist.pt


We need a way to determine how well our model is performing. We will use accuracy as a metric.

In [32]:
accuracy = metrics.Accuracy()

You can set up a single experiment.

- We will show the model batches of 64 images, 
- and for every epoch we will show the model 100 batches (trainsteps=100).
- then, we will test how well the model is doing on unseen data (teststeps=100).
- we will report our results during training to tensorboard, and report all configuration to a toml file.
- we will log the results into a directory called "modellogs", but you could change this to whatever you want.

In [ ]:
import torch
loss_fn = torch.nn.CrossEntropyLoss()

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=500,
    valid_steps=500,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


We will use a very basic model: a model with three linear layers.

In [49]:
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes: int, units1: int, units2: int, units3: int) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.units1 = units1
        self.units2 = units2
        self.units3 = units3
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, units1),
            nn.ReLU(),
            nn.Linear(units1, units2),
            nn.ReLU(),
            nn.Linear(units2, units3),
            nn.ReLU(),
            nn.Linear(units3, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork(
    num_classes=10, units1=256, units2=256, units3=256)

I developped the `tomlserializer` package, it is a useful tool to save configs, models and settings as a tomlfile; that way it is easy to track what you changed during your experiments.

This package will 1. check if there is a `__dict__` attribute available, and if so, it will use that to extract the parameters that do not start with an underscore, like this:

In [35]:
{k: v for k, v in model.__dict__.items() if not k.startswith("_")}

{'training': True, 'num_classes': 10, 'units1': 256, 'units2': 256}

This means that if you want to add more parameters to the `.toml` file, eg `units3`, you can add them to the class like this:

```python
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes: int, units1: int, units2: int, units3: int) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.units1 = units1
        self.units2 = units2
        self.units3 = units3  # <-- add this line
```

And then it will be added to the `.toml` file. Check the result for yourself by using the `.save()` method of the `TomlSerializer` class like this:

In [44]:
tomlserializer = TOMLSerializer()
tomlserializer.save(settings, "settings.toml")
tomlserializer.save(model, "model.toml")

Check the `settings.toml` and `model.toml` files to see what is in there.

You can use the `Trainer` class from my `mltrainer` module to train your model. It has the TOMLserializer integrated, so it will automatically save the settings and model to a toml file if you have added `TOML` as a reporttype in the settings.

In [50]:
trainer = Trainer(
    model=model,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainstreamer,
    validdataloader=validstreamer,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau
)
trainer.loop()

2025-09-17 21:10:33.636 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs\20250917-211033
2025-09-17 21:10:33.637 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 1875/1875 [00:10<00:00, 181.74it/s]
2025-09-17 21:10:44.585 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 0.5053 test 0.4707 metric ['0.8323']
100%|██████████| 1875/1875 [00:15<00:00, 119.25it/s]
2025-09-17 21:11:00.999 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.3659 test 0.4136 metric ['0.8532']
100%|██████████| 1875/1875 [00:17<00:00, 106.28it/s]
2025-09-17 21:11:19.473 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.3316 test 0.3948 metric ['0.8575']
100%|██████████| 3/3 [00:45<00:00, 15.28s/it]


Now, check in the modellogs directory the results of your experiment.

We can now loop this with a naive approach, called a grid-search (why do you think i call it naive?).

In [48]:
units = [256, 128, 64]
for unit1 in units:
    for unit2 in units:
        for unit3 in units:
            print(f"Units: {unit1}, {unit2}, {unit3}")

Units: 256, 256, 256
Units: 256, 256, 128
Units: 256, 256, 64
Units: 256, 128, 256
Units: 256, 128, 128
Units: 256, 128, 64
Units: 256, 64, 256
Units: 256, 64, 128
Units: 256, 64, 64
Units: 128, 256, 256
Units: 128, 256, 128
Units: 128, 256, 64
Units: 128, 128, 256
Units: 128, 128, 128
Units: 128, 128, 64
Units: 128, 64, 256
Units: 128, 64, 128
Units: 128, 64, 64
Units: 64, 256, 256
Units: 64, 256, 128
Units: 64, 256, 64
Units: 64, 128, 256
Units: 64, 128, 128
Units: 64, 128, 64
Units: 64, 64, 256
Units: 64, 64, 128
Units: 64, 64, 64


Of course, this might not be the best way to search for a model; some configurations will be better than others (can you predict up front what will be the best configuration?).

So, feel free to improve upon the gridsearch by adding your own logic.

In [53]:
import torch

units = [256, 128, 64]
loss_fn = torch.nn.CrossEntropyLoss()

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)

for unit1 in units:
    for unit2 in units:
        for unit3 in units:

            model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2, units3=unit3)

            trainer = Trainer(
                model=model,
                settings=settings,
                loss_fn=loss_fn,
                optimizer=optim.Adam,
                traindataloader=trainstreamer,
                validdataloader=validstreamer,
                scheduler=optim.lr_scheduler.ReduceLROnPlateau
            )
            trainer.loop()


2025-09-17 21:20:10.373 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs\20250917-212010
2025-09-17 21:20:10.374 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 1875/1875 [00:11<00:00, 165.22it/s]
2025-09-17 21:20:22.501 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 0.5072 test 0.4250 metric ['0.8493']
100%|██████████| 1875/1875 [00:19<00:00, 95.79it/s]
2025-09-17 21:20:42.992 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.3708 test 0.3899 metric ['0.8566']
100%|██████████| 1875/1875 [00:22<00:00, 84.81it/s]
2025-09-17 21:21:06.070 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.3360 test 0.3596 metric ['0.8713']
100%|██████████| 3/3 [00:55<00:00, 18.56s/it]
2025-09-17 21:21:06.074 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs\20250917-212106
2025-09-17 21:21:06.075 | INFO     | mltrainer.trainer:__init_

In [40]:
import torch

units = [64, 128, 256]
loss_fn = torch.nn.CrossEntropyLoss()

settings = TrainerSettings(
    epochs=8,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)

for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
                model=model,
                settings=settings,
                loss_fn=loss_fn,
                optimizer=optim.Adam,
                traindataloader=trainstreamer,
                validdataloader=validstreamer,
                scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()


2025-09-17 20:50:02.229 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs\20250917-205002
2025-09-17 20:50:02.230 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 1875/1875 [00:04<00:00, 385.01it/s]
2025-09-17 20:50:07.520 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 0.5444 test 0.4426 metric ['0.8447']
100%|██████████| 1875/1875 [00:05<00:00, 342.30it/s]
2025-09-17 20:50:13.389 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.3953 test 0.3981 metric ['0.8562']
100%|██████████| 1875/1875 [00:05<00:00, 316.33it/s]
2025-09-17 20:50:19.691 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.3522 test 0.3770 metric ['0.8628']
100%|██████████| 1875/1875 [00:05<00:00, 328.97it/s]
2025-09-17 20:50:25.790 | INFO     | mltrainer.trainer:report:209 - Epoch 3 train 0.3319 test 0.3681 metric ['0.8698']
100%|██████████| 1875/1875 [00:06<00:00, 296.42it

Because we have set the ReportType to TOML, you will find in every log dir a model.toml and settings.toml file.

Run the experiment, and study the result with tensorboard. 

Locally, it is easy to do that with VS code itself. On the server, you have to take these steps:

- in the terminal, `cd` to the location of the repository
- activate the python environment for the shell. Note how the correct environment is being activated.
- run `tensorboard --logdir=modellogs` in the terminal
- tensorboard will launch at `localhost:6006` and vscode will notify you that the port is forwarded
- you can either press the `launch` button in VScode or open your local browser at `localhost:6006`